In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "features.py").is_file()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import build_features, check_no_leakage, feature_columns, TRAIN_END, VAL_END

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": .3})
OUT = PROJECT_ROOT / "results" / "eda"; OUT.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "brent_clean.csv", parse_dates=["observation_date"])
s = raw.set_index("observation_date")["DCOILBRENTEU"]
raw.describe().T, raw.shape

In [ ]:
print("Số dòng:", len(raw), "| Khoảng:", s.index.min().date(), "->", s.index.max().date())
print("Thiếu:", int(s.isna().sum()), "| Trùng ngày:", int(s.index.duplicated().sum()), "| Giá <= 0:", int((s <= 0).sum()))
gap = s.index.to_series().diff().dt.days
print("Khoảng cách ngày lớn nhất giữa 2 phiên:", int(gap.max()), "ngày")
print(s.describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(s.index, s.values, lw=1, color="black", label="Giá Brent")
for a, b, c, n in [(s.index.min(), TRAIN_END, "tab:blue",   "Train (2015–2022)"),
                   (TRAIN_END,     VAL_END,   "tab:orange", "Validation (2023–2024)"),
                   (VAL_END,       s.index.max(), "tab:green", "Test (01/2025–09/2026)")]:
    ax.axvspan(a, b, color=c, alpha=.25, label=n)
for d in (TRAIN_END, VAL_END):
    ax.axvline(d, color="gray", ls="--", lw=1)
ax.set_title("Giá dầu Brent (USD/thùng), 2015–2026"); ax.set_ylabel("USD/thùng")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, frameon=False)
fig.savefig(OUT/"01_gia_lich_su.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
s.resample("YE").mean().plot.bar(ax=axes[0], title="Giá trung bình theo năm")
axes[0].set_xticklabels([d.year for d in s.resample("YE").mean().index], rotation=45)
s.groupby(s.index.year).agg(["min", "max"]).plot(ax=axes[1], marker="o", title="Giá thấp nhất / cao nhất theo năm")
fig.savefig(OUT/"02_theo_nam.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
r = np.log(s).diff().dropna()
print(r.describe().round(4))
print("Độ lệch (skew):", round(r.skew(), 3), "| Độ nhọn dư (kurtosis):", round(r.kurt(), 3))
print("5 phiên giảm mạnh nhất:\n", r.nsmallest(5).round(3))
print("5 phiên tăng mạnh nhất:\n", r.nlargest(5).round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(r.clip(-.15, .15), bins=100); axes[0].set_title("Phân phối log-return 1 phiên ")
r.rolling(20).std().mul(np.sqrt(252)).plot(ax=axes[1], title="Biến động 20 phiên ")
fig.savefig(OUT/"03_return_volatility.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
def acf(x, nlags=30):
    x = x - x.mean(); d = (x**2).sum()
    return np.array([(x[k:] * x[:len(x)-k]).sum() / d for k in range(1, nlags+1)])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, x, t in [(axes[0], np.log(s).values, "ACF của log-giá"), (axes[1], r.values, "ACF của return")]:
    a = acf(x); ax.bar(range(1, 31), a); ax.axhline(1.96/np.sqrt(len(x)), ls="--", c="r"); ax.axhline(-1.96/np.sqrt(len(x)), ls="--", c="r"); ax.set_title(t)
fig.savefig(OUT/"04_acf.png", dpi=150, bbox_inches="tight"); plt.show()
print("Return trung bình theo thứ trong tuần (%):"); print((r.groupby(r.index.dayofweek).mean()*100).round(3).rename(lambda i: ["T2","T3","T4","T5","T6"][i]))

In [ ]:
feats = build_features(raw)
cols = feature_columns(feats)
print(len(feats), "dòng,", len(cols), "đặc trưng")
print(cols)
feats.head(3).T.head(32)

In [ ]:
for h in (1, 5):
    g = feats[feats[f"split_{h}"] != ""].groupby(f"split_{h}")
    print(f"h={h}"); print(g.agg(n=("price", "size"), obs_min=("observation_date", "min"), obs_max=("observation_date", "max"),
                               tgt_min=(f"target_date_{h}", "min"), tgt_max=(f"target_date_{h}", "max")))
    assert feats.query(f"split_{h}=='train'")[f"target_date_{h}"].max() <= TRAIN_END
    vd = feats.query(f"split_{h}=='val'")[f"target_date_{h}"]
    assert vd.min() > TRAIN_END and vd.max() <= VAL_END
    assert feats.query(f"split_{h}=='test'")[f"target_date_{h}"].min() > VAL_END
print("Ranh giới thời gian hợp lệ.")

In [ ]:
# Kiểm tra ro ri: tinh lai dac trung khi chi co du lieu den t phai ra giong het
check_no_leakage(raw, feats); print("Không rò rỉ thông tin từ tương lai vào đặc trưng.")
# Nhan target_h khop gia that
for h in (1, 5):
    ok = feats.dropna(subset=[f"target_{h}"])
    assert np.allclose(ok[f"target_{h}"].values, s.reindex(ok[f"target_date_{h}"]).values)
print("Nhãn khớp giá thật ở ngày mục tiêu.")

In [ ]:
tr = feats[feats["split_5"] == "train"]
corr = tr[cols].corrwith(tr["target_ret_5"]).sort_values()
fig, ax = plt.subplots(figsize=(8, 7)); corr.plot.barh(ax=ax, title="Tương quan đặc trưng với target_ret_5 (Train)")
fig.savefig(OUT/"05_tuong_quan_dac_trung.png", dpi=150, bbox_inches="tight"); plt.show()
print(corr.abs().sort_values(ascending=False).head(8).round(3))

In [ ]:
cm = tr[["ret_1", "ret_5", "ret_20", "vol_5", "vol_20", "price_ma_20_ratio", "price_ma_50_ratio", "ma_5_20_ratio", "range_20"]].corr()
fig, ax = plt.subplots(figsize=(10, 6)); im = ax.imshow(cm, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm))); ax.set_xticklabels(cm.columns, rotation=60, ha="right"); ax.set_yticks(range(len(cm))); ax.set_yticklabels(cm.columns)
fig.colorbar(im); ax.grid(False); ax.set_title("Ma trận tương quan (nhóm đặc trưng không phụ thuộc mức giá)")
fig.savefig(OUT/"06_ma_tran_tuong_quan.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
for h in (1, 5):
    v = feats[feats[f"split_{h}"] == "val"]
    mae = (v[f"target_{h}"] - v["price"]).abs().mean(); rmse = np.sqrt(((v[f"target_{h}"] - v["price"])**2).mean())
    print(f"Naive h={h} (Validation): MAE={mae:.3f}  RMSE={rmse:.3f} USD")